# Cotton benchmark — silver labeling of 96,080 train tiles

Runs the teacher over all training tiles, applies the box-size filter, then uses
**cross-tile consistency** (overlapping tiles see the same boll twice) to grade
each detection. Writes YOLO-format labels + a consistency report.

**Drive `cotton_bench/` needs:** `cotton_train_pre.zip`, `cotton_train_post.zip`,
`teacher_11s_best.pt`. Runtime: GPU. Expect 1.5–3 h total.

Run the class cells one at a time — each handles one field so a crash costs half.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cotton_bench'
!pip install -q ultralytics
import os
for f in ['cotton_train_pre.zip','cotton_train_post.zip','teacher_11s_best.pt']:
    print(f, 'OK' if os.path.exists(f'{DRIVE}/{f}') else 'MISSING')

In [ ]:
# --- choose which field to process, then run the rest of the notebook ---
CLASS = 'post'      # 'pre' or 'post'
ZIP = f'{DRIVE}/cotton_train_{CLASS}.zip'
OUT = f'{DRIVE}/silver_{CLASS}'
os.makedirs(OUT, exist_ok=True)
!rm -rf /content/train_tiles
!unzip -q "{ZIP}" -d /content/
import glob
tiles = sorted(set(glob.glob('/content/train_tiles/*.jpg') +
                   glob.glob('/content/train_tiles/*.JPG')))
print(CLASS, len(tiles), 'tiles')

In [ ]:
# Teacher inference
from ultralytics import YOLO
import json
from tqdm import tqdm

model = YOLO(f'{DRIVE}/teacher_11s_best.pt')
CONF, IOU, MAX_SIDE = 0.25, 0.45, 120
preds = {}
BATCH = 64
for i in tqdm(range(0, len(tiles), BATCH)):
    batch = tiles[i:i+BATCH]
    res = model.predict(batch, imgsz=640, conf=CONF, iou=IOU,
                        max_det=3000, verbose=False)
    for path, r in zip(batch, res):
        b = r.boxes
        keep = []
        for (x1,y1,x2,y2), c in zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy()):
            if max(x2-x1, y2-y1) <= MAX_SIDE:
                keep.append([float(x1),float(y1),float(x2),float(y2),float(c)])
        preds[os.path.basename(path)] = keep

json.dump(preds, open(f'{OUT}/raw_preds.json','w'))
n = sum(len(v) for v in preds.values())
print(f'{n} boxes over {len(preds)} tiles ({n/len(preds):.0f} per tile)')

In [ ]:
# Cross-tile consistency (fast): grid-bucketed neighbour search
import re
from collections import defaultdict
import numpy as np

TILE = re.compile(r'^(.*?)_r(\d+)_c(\d+)_x(\d+)_y(\d+)', re.I)
CELL = 128          # grid cell size in full-image px
CONS_IOU = 0.4
TILE_SIZE = 640

def parse(name):
    m = TILE.match(name)
    return (m.group(1), int(m.group(4)), int(m.group(5))) if m else None

# group detections by source photo, keeping each box's index within its own tile
by_stem = defaultdict(list)      # stem -> (tile, idx_in_tile, x1, y1, x2, y2)
tiles_of_stem = defaultdict(list)
for tile, boxes in preds.items():
    p = parse(tile)
    if p is None:
        continue
    stem, x0, y0 = p
    tiles_of_stem[stem].append((tile, x0, y0))
    for i, (x1, y1, x2, y2, c) in enumerate(boxes):
        by_stem[stem].append((tile, i, x1 + x0, y1 + y0, x2 + x0, y2 + y0))

verdict = {}
stats = defaultdict(int)

for stem, dets in by_stem.items():
    if not dets:
        continue
    origins = tiles_of_stem[stem]
    ox = np.array([o[1] for o in origins])
    oy = np.array([o[2] for o in origins])

    B = np.array([[d[2], d[3], d[4], d[5]] for d in dets], dtype=np.float32)
    cx = (B[:, 0] + B[:, 2]) / 2
    cy = (B[:, 1] + B[:, 3]) / 2
    area = (B[:, 2] - B[:, 0]) * (B[:, 3] - B[:, 1])

    # how many tiles could see each detection's centre
    nviews = ((cx[:, None] >= ox[None, :]) & (cx[:, None] < ox[None, :] + TILE_SIZE) &
              (cy[:, None] >= oy[None, :]) & (cy[:, None] < oy[None, :] + TILE_SIZE)).sum(1)

    # bucket boxes into a coarse grid; compare only within the 3x3 neighbourhood
    grid = defaultdict(list)
    for i in range(len(dets)):
        grid[(int(cx[i]) // CELL, int(cy[i]) // CELL)].append(i)

    for i, d in enumerate(dets):
        if nviews[i] < 2:
            v = 'single_view'
        else:
            gx, gy = int(cx[i]) // CELL, int(cy[i]) // CELL
            cand = [j for dx in (-1, 0, 1) for dy in (-1, 0, 1)
                    for j in grid.get((gx + dx, gy + dy), ())
                    if dets[j][0] != d[0]]
            v = 'unconfirmed'
            if cand:
                C = B[cand]
                ix1 = np.maximum(B[i, 0], C[:, 0]); iy1 = np.maximum(B[i, 1], C[:, 1])
                ix2 = np.minimum(B[i, 2], C[:, 2]); iy2 = np.minimum(B[i, 3], C[:, 3])
                inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
                union = area[i] + area[cand] - inter
                ious = np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)
                if np.any(ious >= CONS_IOU):
                    v = 'confirmed'
        verdict[(d[0], d[1])] = v      # keyed by (tile, index within that tile)
        stats[v] += 1

tot = sum(stats.values())
for k, n in sorted(stats.items()):
    print(f'{k}: {n} ({n / tot * 100:.1f}%)')

In [ ]:
# Write YOLO labels: keep confirmed + single_view, drop unconfirmed
import shutil
lbl_dir = f'/content/silver_{CLASS}_labels'
os.makedirs(lbl_dir, exist_ok=True)
kept = dropped = 0
for tile, boxes in preds.items():
    lines = []
    for i,(x1,y1,x2,y2,c) in enumerate(boxes):
        if verdict.get((tile,i)) == 'unconfirmed':
            dropped += 1
            continue
        cx, cy = (x1+x2)/2/640, (y1+y2)/2/640
        w, h = (x2-x1)/640, (y2-y1)/640
        lines.append(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
        kept += 1
    with open(f'{lbl_dir}/{os.path.splitext(tile)[0]}.txt','w') as f:
        f.write('\n'.join(lines))
print(f'kept {kept}, dropped {dropped} ({dropped/(kept+dropped)*100:.1f}%)')
shutil.make_archive(f'{OUT}/labels', 'zip', '/content', f'silver_{CLASS}_labels')
print('labels zipped to Drive:', f'{OUT}/labels.zip')

In [ ]:
# Audit sample: 150 random tiles from this field for hand-checking later
import random
random.seed(7)
sample = random.sample(sorted(preds), 150)
os.makedirs('/content/audit', exist_ok=True)
coco = {'images':[], 'annotations':[], 'categories':[{'id':1,'name':'cotton_boll'}]}
aid = 1
for iid, t in enumerate(sorted(sample), 1):
    shutil.copy(f'/content/train_tiles/{t}', f'/content/audit/{t}')
    coco['images'].append({'id':iid,'file_name':t,'width':640,'height':640})
    for i,(x1,y1,x2,y2,c) in enumerate(preds[t]):
        if verdict.get((t,i)) == 'unconfirmed': continue
        coco['annotations'].append({'id':aid,'image_id':iid,'category_id':1,
            'bbox':[x1,y1,x2-x1,y2-y1],'area':(x2-x1)*(y2-y1),'iscrowd':0})
        aid += 1
json.dump(coco, open(f'{OUT}/audit_coco.json','w'))
shutil.make_archive(f'{OUT}/audit_tiles','zip','/content','audit')
print(f'audit set: 150 tiles, {aid-1} boxes -> Drive/{OUT}')

In [ ]:
import json
CLASS = 'pre'
OUT = f'{DRIVE}/silver_{CLASS}'
preds = json.load(open(f'{OUT}/raw_preds.json'))
print(len(preds), 'tiles loaded')

In [ ]:
import json, glob, os
import numpy as np

DRIVE = '/content/drive/MyDrive/cotton_bench'
rows = []

for CLASS in ['pre', 'post']:
    preds = json.load(open(f'{DRIVE}/silver_{CLASS}/raw_preds.json'))
    n_tiles = len(preds)
    n_raw = sum(len(v) for v in preds.values())

    # count labels actually written
    lbls = glob.glob(f'/content/silver_{CLASS}_labels/*.txt')
    if lbls:
        n_kept = sum(sum(1 for _ in open(f)) for f in lbls)
    else:
        n_kept = None   # label folder not in this session

    per_tile = [len(v) for v in preds.values()]
    rows.append(dict(cls=CLASS, tiles=n_tiles, raw=n_raw, kept=n_kept,
                     mean=np.mean(per_tile), med=np.median(per_tile),
                     p90=np.percentile(per_tile, 90), mx=max(per_tile),
                     empty=sum(1 for p in per_tile if p == 0)))

print(f"{'field':<8}{'tiles':>8}{'raw boxes':>12}{'per tile':>10}"
      f"{'median':>9}{'p90':>7}{'max':>7}{'empty':>8}")
for r in rows:
    print(f"{r['cls']:<8}{r['tiles']:>8}{r['raw']:>12,}{r['mean']:>10.0f}"
          f"{r['med']:>9.0f}{r['p90']:>7.0f}{r['mx']:>7}{r['empty']:>8}")

tot_tiles = sum(r['tiles'] for r in rows)
tot_raw = sum(r['raw'] for r in rows)
print(f"\nTOTAL: {tot_tiles:,} tiles, {tot_raw:,} raw detections")
print("consistency: pre 60.5% confirmed / 33.6% single-view / 5.9% dropped")
print("             post 61.4% confirmed / 34.4% single-view / 4.2% dropped")
print(f"labels kept: pre 2,006,694 | post 13,713,954 | total 15,720,648")